# Data Ingestion – Reproducible Pipeline

Ingests raw documents from `corpus.csv` into Qdrant using **automated chunking**.

| Parameter | Value |
|---|---|
| Chunking | `RecursiveCharacterTextSplitter` |
| Chunk size | 1024 tokens |
| Chunk overlap | 128 tokens |
| Dense embedding | OpenAI `text-embedding-3-small` |
| Sparse embedding | BM25 (fastembed `Qdrant/bm25`) |
| Retrieval mode | Hybrid (dense + sparse) |
| Collection | `method_naive_chunks_chunk_size_1024_chunk_overlap_128_hybrid` |

### When to re-run
- Corpus documents changed
- Embedding model changed
- Qdrant data lost

In [1]:
import pandas as pd
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_qdrant import RetrievalMode

from config import settings, RAG_COLLECTION
from components.vector_store import get_vector_store, delete_vector_store

CHUNK_SIZE = 1024
CHUNK_OVERLAP = 128

print(f"Collection: {RAG_COLLECTION}")
print(f"Chunking: RecursiveCharacterTextSplitter({CHUNK_SIZE}/{CHUNK_OVERLAP})")

dense_embedding: client=<openai.resources.embeddings.Embeddings object at 0x7fa86824f710> async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x7fa86825c2d0> model='text-embedding-3-small' dimensions=None deployment='text-embedding-ada-002' openai_api_version=None openai_api_base=None openai_api_type=None openai_proxy=None embedding_ctx_length=8191 openai_api_key=SecretStr('**********') openai_organization=None allowed_special=None disallowed_special=None chunk_size=1000 max_retries=2 request_timeout=None headers=None tiktoken_enabled=True tiktoken_model_name=None show_progress_bar=False model_kwargs={} skip_empty=False default_headers=None default_query=None retry_min_seconds=4 retry_max_seconds=20 http_client=None http_async_client=None check_embedding_ctx_length=True
Collection: method_naive_chunks_chunk_size_1024_chunk_overlap_128_hybrid
Chunking: RecursiveCharacterTextSplitter(1024/128)


In [2]:
corpus_df = pd.read_csv("dataset/corpus.csv")
print(f"Loaded {len(corpus_df)} documents from corpus.csv")
print(f"Columns: {list(corpus_df.columns)}")

documents = []
for idx, row in corpus_df.iterrows():
    doc = Document(
        page_content=row["name"] + "\n" + row["full_text"],
        metadata={"document_id": idx},
    )
    documents.append(doc)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
chunks = splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks")

Loaded 498 documents from corpus.csv
Columns: ['name', 'full_text']
Split into 1072 chunks


In [3]:
delete_vector_store(RAG_COLLECTION)

Collection 'method_naive_chunks_chunk_size_1024_chunk_overlap_128_hybrid' deleted.


In [4]:
vs = get_vector_store(
    mode=RetrievalMode.HYBRID,
    collection_name=RAG_COLLECTION,
)
vs.add_documents(chunks)
print(f"Ingested {len(chunks)} chunks into '{RAG_COLLECTION}'")

Collection 'method_naive_chunks_chunk_size_1024_chunk_overlap_128_hybrid' not found. Creating for RetrievalMode.HYBRID mode...
Ingested 1072 chunks into 'method_naive_chunks_chunk_size_1024_chunk_overlap_128_hybrid'


In [5]:
from qdrant_client import QdrantClient

client = QdrantClient(url=settings.qdrant_url)
info = client.get_collection(RAG_COLLECTION)
print(f"{RAG_COLLECTION}: {info.points_count} points")

method_naive_chunks_chunk_size_1024_chunk_overlap_128_hybrid: 1072 points
